# 03 — Runtime 12-feature Model, Optuna Sensitivity & Paper Outputs

Audits the **runtime-oriented 12-feature** model (`unified_relative_shape_v2__lgbm`,
corrected USBVPN artifact), the **Optuna sensitivity** run, the **simulation-only** decision
layer, and produces final publication-ready replacement tables/figures.

**Integrity rule:** no numbers are invented. Values are loaded from saved artifacts
(`final_metrics.json`, per-flow score CSVs, Optuna outputs) or recomputed from saved
per-flow scores. Anything not reconstructable is emitted as a **MISSING / NEEDS MANUAL INPUT**
cell naming the required file.

**Corrected runtime artifact:** `artifacts/unified_feature_contract_v2_corrected_usbvpn/`
(supersedes the archived original). Label encoding: **1 = VPN, 0 = non-VPN**.

> All firewall-style outputs (PASS / FLAG_REVIEW / SIMULATED_BLOCK) are **simulation-only**.
> The runtime model is **not** described as deployment-ready.


In [1]:

# --- Setup ---
import os, sys, json, warnings
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
try:
    import seaborn as sns; sns.set_theme(style="whitegrid"); HAS_SNS=True
except Exception: HAS_SNS=False

RNG_SEED=42; np.random.seed(RNG_SEED)
ROOT=Path.cwd().resolve()
while not ((ROOT/"artifacts").exists() and (ROOT/"data").exists()) and ROOT!=ROOT.parent: ROOT=ROOT.parent
print("PROJECT ROOT:",ROOT)
OUT=ROOT/"paper_audit_outputs"; TBL,FIG,MET,LOG=OUT/"tables",OUT/"figures",OUT/"metrics",OUT/"logs"
for d in (TBL,FIG,MET,LOG): d.mkdir(parents=True,exist_ok=True)
ART=ROOT/"artifacts"/"unified_feature_contract_v2_corrected_usbvpn"
MODELDIR=ART/"models"/"unified_relative_shape_v2__lgbm"
OPT=ART/"optuna_corrected_sensitivity"; SAN=ART/"sanity_checks"

MISSING=[]; GENERATED=[]
def mark_missing(section,what,needed):
    MISSING.append({"section":section,"what":what,"needed":needed})
    print(f"  [MISSING/{section}] {what}  -> needs: {needed}")
def save_table(df,name):
    p=TBL/name; df.to_csv(p,index=False); GENERATED.append(str(p.relative_to(ROOT)))
    print(f"  table: {p.relative_to(ROOT)} ({df.shape[0]}x{df.shape[1]})"); return p
def save_fig(fig,name,dpi=300):
    p=FIG/name; fig.savefig(p,dpi=dpi,bbox_inches="tight"); plt.close(fig)
    GENERATED.append(str(p.relative_to(ROOT))); print(f"  figure: {p.relative_to(ROOT)} (dpi={dpi})"); return p
def save_metric(obj,name):
    p=MET/name; p.write_text(json.dumps(obj,indent=2,default=str),encoding="utf-8")
    GENERATED.append(str(p.relative_to(ROOT))); print(f"  metric: {p.relative_to(ROOT)}"); return p
def save_table_all(df,basename,caption=None,index=False):
    """Write CSV + Markdown + LaTeX versions of a table."""
    save_table(df, basename+".csv")
    try:
        (TBL/(basename+".md")).write_text(df.to_markdown(index=index),encoding="utf-8")
        GENERATED.append(str((TBL/(basename+".md")).relative_to(ROOT)))
    except Exception as e: print("   md fail",e)
    try:
        (TBL/(basename+".tex")).write_text(df.to_latex(index=index,escape=True,
            caption=caption or basename, label="tab:"+basename, float_format="%.5f"),encoding="utf-8")
        GENERATED.append(str((TBL/(basename+".tex")).relative_to(ROOT)))
    except Exception as e: print("   tex fail",e)
def load_json(p):
    p=Path(p)
    if not p.exists(): return None
    return json.loads(p.read_text(encoding="utf-8"))
print("setup ready | artifact exists:",ART.exists())


PROJECT ROOT: C:\Users\scoti\PycharmProjects\ai-vpn-firewall
setup ready | artifact exists: True


## 1. Load previous audit outputs (Notebook 1 & 2)

In [2]:

REQUIRED_PREV = [
 "tables/dataset_composition.csv","tables/capture_statistics_extended.csv",
 "tables/within_dataset_split_summary.csv","tables/within_dataset_leakage_check.csv",
 "tables/lodo_composition.csv","tables/feature_definitions_21.csv",
 "tables/feature_summary_21_by_dataset.csv","tables/flow_construction_audit.csv",
 "tables/intra_dataset_metrics.csv","tables/lodo_metrics.csv",
 "tables/construction_scale_with_iqr.csv","tables/dataset_fingerprinting_metrics.csv",
 "tables/feature_instability_detailed.csv","tables/instability_verdict_features.csv",
 "tables/preprocessing_sensitivity.csv","tables/robustness_checks_diagnostic.csv",
 "tables/nb2_data_composition.csv","tables/nb2_split_leakage_check.csv",
]
prev={}; status_rows=[]
for rel in REQUIRED_PREV:
    p=OUT/rel; ok=p.exists()
    status_rows.append({"file":rel,"exists":ok,"rows":(pd.read_csv(p).shape[0] if ok else None)})
    if ok:
        try: prev[Path(rel).stem]=pd.read_csv(p)
        except Exception as e: print("  read fail",rel,e)
    else:
        mark_missing("1",f"previous output missing: {rel}","run Notebook 1/2")
status_df=pd.DataFrame(status_rows)
save_table(status_df,"nb3_prev_output_status.csv")
print("present:",int(status_df.exists.sum()),"/",len(status_df))
status_df


  table: paper_audit_outputs\tables\nb3_prev_output_status.csv (18x3)
present: 18 / 18


,file,exists,rows
0,tables/dataset_composition.csv,True,3
1,tables/capture_statistics_extended.csv,True,3
2,tables/within_dataset_split_summary.csv,True,9
3,tables/within_dataset_leakage_check.csv,True,3
4,tables/lodo_composition.csv,True,3
5,tables/feature_definitions_21.csv,True,21
6,tables/feature_summary_21_by_dataset.csv,True,42
7,tables/flow_construction_audit.csv,True,3
8,tables/intra_dataset_metrics.csv,True,3
9,tables/lodo_metrics.csv,True,3


## 2. Runtime 12-feature contract audit

`unified_relative_shape_v2` — 12 relative/shape features, deliberately scale-invariant to
mitigate the construction-scale mismatch identified in Notebook 2.


In [3]:

RUNTIME_FEATURES = ["sz_cv","sz_iqr","sz_qratio","sz_median_to_mean","sz_p25_median_ratio",
    "sz_p75_median_ratio","sz_iqr_norm_median","iat_cv","iat_iqr",
    "direction_balance_bytes","direction_balance_packets","dispersion_symmetry"]
FEATURE_GROUPS = {
 "sz_cv":"size_shape","sz_iqr":"size_shape","sz_qratio":"size_shape","sz_median_to_mean":"size_shape",
 "sz_p25_median_ratio":"size_shape","sz_p75_median_ratio":"size_shape","sz_iqr_norm_median":"size_shape",
 "iat_cv":"timing_shape","iat_iqr":"timing_shape",
 "direction_balance_bytes":"directionality","direction_balance_packets":"directionality",
 "dispersion_symmetry":"dispersion_symmetry"}
fo = load_json(MODELDIR/"feature_order.json")
assert fo and fo["features"]==RUNTIME_FEATURES, "feature_order.json mismatch!"
groups_df=pd.DataFrame({"feature":RUNTIME_FEATURES,
    "group":[FEATURE_GROUPS[f] for f in RUNTIME_FEATURES],
    "family":"unified_relative_shape_v2"})
save_table_all(groups_df,"runtime_feature_groups","Runtime 12-feature groups")
groups_df


  table: paper_audit_outputs\tables\runtime_feature_groups.csv (12x3)


,feature,group,family
0,sz_cv,size_shape,unified_relative_shape_v2
1,sz_iqr,size_shape,unified_relative_shape_v2
2,sz_qratio,size_shape,unified_relative_shape_v2
3,sz_median_to_mean,size_shape,unified_relative_shape_v2
4,sz_p25_median_ratio,size_shape,unified_relative_shape_v2
5,sz_p75_median_ratio,size_shape,unified_relative_shape_v2
6,sz_iqr_norm_median,size_shape,unified_relative_shape_v2
7,iat_cv,timing_shape,unified_relative_shape_v2
8,iat_iqr,timing_shape,unified_relative_shape_v2
9,direction_balance_bytes,directionality,unified_relative_shape_v2


In [4]:

# --- 2b. runtime dataset composition + schema validation ---
RT=pd.read_parquet(ART/"data"/"unified_flows.parquet")
RT[RUNTIME_FEATURES]=RT[RUNTIME_FEATURES].replace([np.inf,-np.inf],np.nan)
comp_rows=[]; both_classes={}
for ds,g in RT.groupby("dataset"):
    vpn=int((g.label==1).sum()); non=int((g.label==0).sum())
    comp_rows.append({"dataset":ds,"flows":len(g),"vpn":vpn,"nonvpn":non,
        "vpn_pct":round(100*vpn/len(g),3),"captures":int(g.capture_id.nunique())})
    both_classes[ds]=bool(vpn>0 and non>0)
for sp,g in RT.groupby("split"):
    comp_rows.append({"dataset":f"__split:{sp}","flows":len(g),"vpn":int((g.label==1).sum()),
        "nonvpn":int((g.label==0).sum()),"vpn_pct":round(100*(g.label==1).mean(),3),
        "captures":int(g.capture_id.nunique())})
rt_comp=pd.DataFrame(comp_rows)
save_table_all(rt_comp,"runtime_dataset_composition","Runtime artifact dataset composition")
print("all datasets have both classes:",both_classes)
assert all(both_classes.values()), "a runtime dataset is single-class!"
rt_comp


  table: paper_audit_outputs\tables\runtime_dataset_composition.csv (6x6)
all datasets have both classes: {'iscx': True, 'usbvpn': True, 'vnat': True}


,dataset,flows,vpn,nonvpn,vpn_pct,captures
0,iscx,11801,2943,8858,24.939,140
1,usbvpn,50759,8456,42303,16.659,511
2,vnat,8107,374,7733,4.613,165
3,__split:test,13707,2001,11706,14.598,138
4,__split:train,46980,7769,39211,16.537,556
5,__split:val,9980,2003,7977,20.070,122


In [5]:

# --- 2c. runtime feature summary (NaN/inf/invalid + distribution stats) ---
raw=pd.read_parquet(ART/"data"/"unified_flows.parquet")  # un-replaced to count true inf
sum_rows=[]
for f in RUNTIME_FEATURES:
    col=raw[f].astype(float)
    sum_rows.append({"feature":f,"dtype":str(raw[f].dtype),
        "n_missing":int(col.isna().sum()),
        "n_inf":int(np.isinf(col).sum()),
        "n_invalid_neg_where_ratio":int((col<0).sum()) if f.endswith("ratio") or "balance" in f else 0,
        "min":float(np.nanmin(col.replace([np.inf,-np.inf],np.nan))),
        "q1":float(np.nanpercentile(col.replace([np.inf,-np.inf],np.nan),25)),
        "median":float(np.nanmedian(col.replace([np.inf,-np.inf],np.nan))),
        "mean":float(np.nanmean(col.replace([np.inf,-np.inf],np.nan))),
        "q3":float(np.nanpercentile(col.replace([np.inf,-np.inf],np.nan),75)),
        "max":float(np.nanmax(col.replace([np.inf,-np.inf],np.nan))),
        "std":float(np.nanstd(col.replace([np.inf,-np.inf],np.nan)))})
rt_sum=pd.DataFrame(sum_rows)
save_table_all(rt_sum,"runtime_feature_summary","Runtime 12-feature summary statistics")
rt_sum[["feature","n_missing","n_inf","median","mean","std"]]


  table: paper_audit_outputs\tables\runtime_feature_summary.csv (12x12)


,feature,n_missing,n_inf,median,mean,std
0,sz_cv,0,0,1.081201,0.968287,0.540739
1,sz_iqr,0,0,441.250000,687.178428,787.148546
2,sz_qratio,0,0,8.610577,11.657679,11.913807
3,sz_median_to_mean,0,0,0.372810,0.579922,0.436724
4,sz_p25_median_ratio,0,0,0.812500,0.654774,0.329570
5,sz_p75_median_ratio,0,0,2.530612,4.891063,4.992246
6,sz_iqr_norm_median,0,0,1.928669,4.236289,4.973100
7,iat_cv,0,0,2.176207,5.234683,7.601396
8,iat_iqr,0,0,0.002868,80.393727,835.950098
9,direction_balance_bytes,0,0,-0.449664,-0.238493,0.646382


In [6]:

# --- 2d. count difference vs 21-feature diagnostic dataset ---
diag=prev.get("nb2_data_composition")
diff_rows=[]
if diag is not None:
    dg=diag.groupby("dataset")[["flows","vpn","nonvpn"]].sum()
    for ds in ["iscx","usbvpn","vnat"]:
        rg=RT[RT.dataset==ds]
        rt_flows=len(rg); rt_vpn=int((rg.label==1).sum()); rt_non=int((rg.label==0).sum())
        dflows=int(dg.loc[ds,"flows"]); dvpn=int(dg.loc[ds,"vpn"]); dnon=int(dg.loc[ds,"nonvpn"])
        diff_rows.append({"dataset":ds,
            "diag21_flows":dflows,"runtime12_flows":rt_flows,"flow_delta":dflows-rt_flows,
            "diag21_vpn":dvpn,"runtime12_vpn":rt_vpn,"vpn_delta":dvpn-rt_vpn,
            "diag21_nonvpn":dnon,"runtime12_nonvpn":rt_non,"nonvpn_delta":dnon-rt_non})
    diff_df=pd.DataFrame(diff_rows)
    # reconstructable explanation
    expl=[]
    for _,r in diff_df.iterrows():
        if r.flow_delta==0:
            expl.append("identical counts: same flows entered both representations")
        elif r.vpn_delta==0 and r.nonvpn_delta!=0:
            expl.append(f"VPN identical ({r.runtime12_vpn}); difference is {r.nonvpn_delta} nonVPN flows only "
                        f"-> runtime corrected-USBVPN rebuild applied different nonVPN inclusion/sampling. "
                        f"Exact per-reason breakdown NOT logged.")
        else:
            expl.append("flows differ in both classes; per-reason breakdown not logged.")
    diff_df["explanation"]=expl
    save_table_all(diff_df,"runtime_count_difference_explanation","Runtime vs diagnostic count differences")
    mark_missing("2","exact per-reason breakdown of USBVPN nonVPN count delta between diagnostic(21-feat) "
                 "and runtime(12-feat) artifacts","build_and_train_corrected_usbvpn.py nonVPN selection logs / row-level provenance")
    print(diff_df.to_string(index=False))
else:
    mark_missing("2","21-feature diagnostic composition unavailable for comparison","Notebook 2 nb2_data_composition.csv")
diff_df if diag is not None else None


  table: paper_audit_outputs\tables\runtime_count_difference_explanation.csv (3x11)
  [MISSING/2] exact per-reason breakdown of USBVPN nonVPN count delta between diagnostic(21-feat) and runtime(12-feat) artifacts  -> needs: build_and_train_corrected_usbvpn.py nonVPN selection logs / row-level provenance
dataset  diag21_flows  runtime12_flows  flow_delta  diag21_vpn  runtime12_vpn  vpn_delta  diag21_nonvpn  runtime12_nonvpn  nonvpn_delta                                                                                                                                                                        explanation
   iscx         11801            11801           0        2943           2943          0           8858              8858             0                                                                                                                          identical counts: same flows entered both representations
 usbvpn         52704            50759        1945        8456   

,dataset,diag21_flows,runtime12_flows,flow_delta,diag21_vpn,runtime12_vpn,vpn_delta,diag21_nonvpn,runtime12_nonvpn,nonvpn_delta,explanation
0,iscx,11801,11801,0,2943,2943,0,8858,8858,0,identical counts: same flows entered both repr...
1,usbvpn,52704,50759,1945,8456,8456,0,44248,42303,1945,VPN identical (8456); difference is 1945 nonVP...
2,vnat,8107,8107,0,374,374,0,7733,7733,0,identical counts: same flows entered both repr...


## 3. Runtime LightGBM candidate audit

Fixed runtime candidate (verified in `build_and_train_corrected_usbvpn.py`):
`LGBMClassifier(objective="binary", n_estimators=500, learning_rate=0.05, num_leaves=31,
min_child_samples=20, subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.1,
scale_pos_weight=n_neg/n_pos, random_state=42)` with isotonic calibration.

Metrics are recomputed from the **saved per-flow scores** and cross-checked against the
artifact's `final_metrics.json`. LODO AUCs are recomputed from saved raw LODO scores.
Decision-state counts use the saved review/block thresholds — **simulation-only**.


In [7]:

import joblib
fm=load_json(ART/"final_metrics.json")
thr=load_json(MODELDIR/"thresholds.json")
review_thr=thr["review_threshold"]; block_thr=thr["block_threshold"]

# pooled test scores (isotonic-calibrated), per flow
pooled=pd.read_csv(SAN/"pooled_test_scores_by_dataset.csv")
y=pooled.label.values.astype(int); s=pooled.calibrated_score.values.astype(float)
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
def ece(y,p,bins=10):
    y=np.asarray(y);p=np.asarray(p);edges=np.linspace(0,1,bins+1);e=0.0
    for i in range(bins):
        m=(p>=edges[i])&(p<edges[i+1] if i<bins-1 else p<=edges[i+1])
        if m.sum()==0: continue
        e+=(m.sum()/len(p))*abs(p[m].mean()-y[m].mean())
    return float(e)
pooled_roc=float(roc_auc_score(y,s)); pooled_pr=float(average_precision_score(y,s))
pooled_ece=ece(y,s); pooled_brier=float(brier_score_loss(y,s))
print(f"pooled (recomputed from saved scores): ROC-AUC={pooled_roc:.5f} PR-AUC={pooled_pr:.5f} "
      f"ECE={pooled_ece:.5f} Brier={pooled_brier:.5f}")
print(f"artifact final_metrics: ROC-AUC={fm['test_auc']:.5f} PR-AUC={fm['test_pr_auc']:.5f} ECE={fm['test_ece']:.5f}")
assert abs(pooled_roc-fm['test_auc'])<1e-3, "recomputed pooled ROC-AUC disagrees with artifact!"


pooled (recomputed from saved scores): ROC-AUC=0.99138 PR-AUC=0.96590 ECE=0.04501 Brier=0.03121
artifact final_metrics: ROC-AUC=0.99138 PR-AUC=0.96590 ECE=0.30951


In [8]:

# --- 3b. LODO AUC recomputed from saved raw LODO scores ---
lodo_scores=pd.read_csv(SAN/"lodo_scores_by_dataset.csv")
lodo_rt={}
for ds,g in lodo_scores.groupby("held_out_dataset"):
    yy=g.label.values.astype(int); ss=g.lodo_raw_score.values.astype(float)
    lodo_rt[ds]=float(roc_auc_score(yy,ss)) if len(np.unique(yy))>1 else np.nan
lodo_mean=float(np.nanmean(list(lodo_rt.values()))); lodo_min=float(np.nanmin(list(lodo_rt.values())))
print("LODO (recomputed):",{k:round(v,5) for k,v in lodo_rt.items()},"mean",round(lodo_mean,5),"min",round(lodo_min,5))
print("LODO (artifact)  :",{ "iscx":fm["lodo_iscx_auc"],"usbvpn":fm["lodo_usbvpn_auc"],"vnat":fm["lodo_vnat_auc"]})
for ds in lodo_rt:
    assert abs(lodo_rt[ds]-fm[f"lodo_{ds}_auc"])<1e-3, f"LODO {ds} mismatch"
domain_auc=fm["domain_auc"]  # artifact-sourced (RandomForest domain classifier in build script)
print("domain_auc (artifact-sourced):",domain_auc)


LODO (recomputed): {'iscx': 0.45279, 'usbvpn': 0.48018, 'vnat': 0.98064} mean 0.63787 min 0.45279
LODO (artifact)  : {'iscx': 0.45278762289616536, 'usbvpn': 0.48017997557997555, 'vnat': 0.9806374804422442}
domain_auc (artifact-sourced): 0.9589283980028643


In [9]:

# --- 3c. decision-state distribution + per-state FPR (SIMULATION-ONLY) ---
def decision_state(score):
    if score < review_thr: return "PASS"
    if score < block_thr:  return "FLAG_REVIEW"
    return "SIMULATED_BLOCK"
pooled=pooled.copy(); pooled["state"]=[decision_state(x) for x in s]
state_rows=[]
for st in ["PASS","FLAG_REVIEW","SIMULATED_BLOCK"]:
    sub=pooled[pooled.state==st]
    n=len(sub); npos=int((sub.label==1).sum()); nneg=int((sub.label==0).sum())
    # FPR contribution = nonVPN routed to this state / all nonVPN
    fpr_state=nneg/int((pooled.label==0).sum()) if (pooled.label==0).sum() else np.nan
    state_rows.append({"decision_state":st,"flows":n,"vpn":npos,"nonvpn":nneg,
        "share_of_flows":round(n/len(pooled),5),
        "nonvpn_routed_here_rate":round(fpr_state,5),
        "vpn_routed_here_rate":round(npos/int((pooled.label==1).sum()),5)})
state_df=pd.DataFrame(state_rows)
state_df["NOTE"]="SIMULATION-ONLY decision layer; not deployment-ready"
save_table_all(state_df,"runtime_decision_state_distribution","Simulation-only decision-state distribution (pooled test)")
print(state_df.to_string(index=False))

# simulation threshold table (review + block confusion at pooled level) from artifact-saved per_dataset_threshold_metrics
thr_metrics=pd.read_csv(SAN/"per_dataset_threshold_metrics.csv")
save_table_all(thr_metrics,"runtime_simulation_threshold_table","Simulation thresholds: per-dataset review/block metrics")
state_df


  table: paper_audit_outputs\tables\runtime_decision_state_distribution.csv (3x8)
 decision_state  flows  vpn  nonvpn  share_of_flows  nonvpn_routed_here_rate  vpn_routed_here_rate                                                 NOTE
           PASS  10324   57   10267         0.75319                  0.87707               0.02849 SIMULATION-ONLY decision layer; not deployment-ready
    FLAG_REVIEW    571   12     559         0.04166                  0.04775               0.00600 SIMULATION-ONLY decision layer; not deployment-ready
SIMULATED_BLOCK   2812 1932     880         0.20515                  0.07518               0.96552 SIMULATION-ONLY decision layer; not deployment-ready
  table: paper_audit_outputs\tables\runtime_simulation_threshold_table.csv (4x19)


,decision_state,flows,vpn,nonvpn,share_of_flows,nonvpn_routed_here_rate,vpn_routed_here_rate,NOTE
0,PASS,10324,57,10267,0.75319,0.87707,0.02849,SIMULATION-ONLY decision layer; not deployment...
1,FLAG_REVIEW,571,12,559,0.04166,0.04775,0.00600,SIMULATION-ONLY decision layer; not deployment...
2,SIMULATED_BLOCK,2812,1932,880,0.20515,0.07518,0.96552,SIMULATION-ONLY decision layer; not deployment...


In [10]:

# --- 3d. runtime_metrics.csv (consolidated) ---
rt_metrics=pd.DataFrame([{
    "model":"unified_relative_shape_v2__lgbm","artifact_version":fm["artifact_version"],
    "n_features":12,"pooled_test_roc_auc":pooled_roc,"pooled_test_pr_auc":pooled_pr,
    "pooled_test_ece":pooled_ece,"pooled_test_brier":pooled_brier,
    "lodo_iscx":lodo_rt["iscx"],"lodo_usbvpn":lodo_rt["usbvpn"],"lodo_vnat":lodo_rt["vnat"],
    "lodo_mean":lodo_mean,"lodo_min":lodo_min,"domain_auc":domain_auc,
    "review_threshold":review_thr,"block_threshold":block_thr,
    "pass_count":int((pooled.state=="PASS").sum()),
    "flag_review_count":int((pooled.state=="FLAG_REVIEW").sum()),
    "simulated_block_count":int((pooled.state=="SIMULATED_BLOCK").sum()),
}])
save_table_all(rt_metrics,"runtime_metrics","Runtime 12-feature model metrics (simulation-only decision layer)")
save_metric(rt_metrics.iloc[0].to_dict(),"runtime_metrics.json")
rt_metrics.T


  table: paper_audit_outputs\tables\runtime_metrics.csv (1x18)
  metric: paper_audit_outputs\metrics\runtime_metrics.json


,0
model,unified_relative_shape_v2__lgbm
artifact_version,corrected_usbvpn_v2
n_features,12
pooled_test_roc_auc,0.991375
pooled_test_pr_auc,0.9659
pooled_test_ece,0.04501
pooled_test_brier,0.031205
lodo_iscx,0.452788
lodo_usbvpn,0.48018
lodo_vnat,0.980637


In [11]:

# --- 3e. calibration plot + score distributions + decision-state plot ---
from sklearn.calibration import calibration_curve
fig,ax=plt.subplots(figsize=(6,5))
frac_pos,mean_pred=calibration_curve(y,s,n_bins=10,strategy="uniform")
ax.plot([0,1],[0,1],"k--",lw=0.8,label="perfect")
ax.plot(mean_pred,frac_pos,"o-",label=f"runtime (ECE={pooled_ece:.3f}, Brier={pooled_brier:.3f})")
ax.set(title="Runtime calibration curve (pooled test, calibrated)",xlabel="mean predicted",ylabel="empirical frequency"); ax.legend()
save_fig(fig,"runtime_calibration_curve.png")

fig,ax=plt.subplots(1,3,figsize=(15,4),sharey=True)
for axc,ds in zip(ax,["iscx","usbvpn","vnat"]):
    sub=pooled[pooled.dataset==ds]
    axc.hist(sub[sub.label==0].calibrated_score,bins=40,alpha=0.6,label="nonVPN",density=True)
    axc.hist(sub[sub.label==1].calibrated_score,bins=40,alpha=0.6,label="VPN",density=True)
    axc.axvline(review_thr,color="orange",ls="--",lw=0.8); axc.axvline(block_thr,color="red",ls="--",lw=0.8)
    axc.set(title=ds,xlabel="calibrated score"); axc.legend()
fig.suptitle("Runtime score distribution by class/dataset (orange=review, red=block)")
save_fig(fig,"runtime_score_distribution.png")

fig,ax=plt.subplots(figsize=(7,4))
piv=pooled.groupby(["dataset","state"]).size().unstack(fill_value=0)[["PASS","FLAG_REVIEW","SIMULATED_BLOCK"]]
piv.plot(kind="bar",stacked=True,ax=ax,color=["#4c9f70","#e8a33d","#c0392b"])
ax.set(title="Simulation-only decision states by dataset (pooled test)",ylabel="flows"); ax.tick_params(axis="x",rotation=0)
save_fig(fig,"runtime_decision_state_distribution.png")


  figure: paper_audit_outputs\figures\runtime_calibration_curve.png (dpi=300)


  figure: paper_audit_outputs\figures\runtime_score_distribution.png (dpi=300)
  figure: paper_audit_outputs\figures\runtime_decision_state_distribution.png (dpi=300)


WindowsPath('C:/Users/scoti/PycharmProjects/ai-vpn-firewall/paper_audit_outputs/figures/runtime_decision_state_distribution.png')

## 4. Optuna sensitivity audit

Loaded from `optuna_corrected_sensitivity/`. The Optuna run is a **post-hoc sensitivity
check** on the corrected artifact (60 trials, `TPESampler(seed=42)`, objective = validation
ROC-AUC, source/validation only). It is **not** an automatic replacement for the fixed model.


In [12]:

opt_pooled=load_json(OPT/"optuna_corrected_pooled_metrics.json")
opt_best=load_json(OPT/"optuna_corrected_best_params.json")
opt_lodo=pd.read_csv(OPT/"optuna_corrected_lodo_metrics.csv")
opt_lodo_map={r.held_out:r.lodo_roc_auc for _,r in opt_lodo.iterrows()}
opt_lodo_mean=float(np.nanmean(list(opt_lodo_map.values()))); opt_lodo_min=float(np.nanmin(list(opt_lodo_map.values())))

fixed_vs=pd.DataFrame([
 {"metric":"test_roc_auc","fixed":fm["test_auc"],"optuna":opt_pooled["test_roc_auc"]},
 {"metric":"test_pr_auc","fixed":fm["test_pr_auc"],"optuna":opt_pooled["test_pr_auc"]},
 {"metric":"test_ece","fixed":fm["test_ece"],"optuna":opt_pooled["test_ece_calibrated"]},
 {"metric":"lodo_iscx","fixed":fm["lodo_iscx_auc"],"optuna":opt_lodo_map.get("iscx")},
 {"metric":"lodo_usbvpn","fixed":fm["lodo_usbvpn_auc"],"optuna":opt_lodo_map.get("usbvpn")},
 {"metric":"lodo_vnat","fixed":fm["lodo_vnat_auc"],"optuna":opt_lodo_map.get("vnat")},
 {"metric":"lodo_mean","fixed":fm["lodo_mean_auc"],"optuna":opt_lodo_mean},
 {"metric":"lodo_min","fixed":fm["lodo_min_auc"],"optuna":opt_lodo_min},
])
fixed_vs["optuna_minus_fixed"]=fixed_vs["optuna"]-fixed_vs["fixed"]
# direction: for ECE lower is better, others higher is better
def better(m,d):
    if m=="test_ece": return "optuna better" if d<0 else ("fixed better" if d>0 else "tie")
    return "optuna better" if d>0 else ("fixed better" if d<0 else "tie")
fixed_vs["interpretation"]=[better(m,d) for m,d in zip(fixed_vs.metric,fixed_vs.optuna_minus_fixed)]
save_table_all(fixed_vs,"fixed_vs_optuna","Fixed vs Optuna sensitivity (corrected artifact)")
print(fixed_vs.to_string(index=False))
opt_best_params=opt_best["pooled"]
save_metric({"best_val_auc":opt_pooled["best_val_auc"],"best_params_pooled":opt_best_params,
             "lodo_best_params":opt_best["lodo"]},"optuna_best_params_summary.json")


  table: paper_audit_outputs\tables\fixed_vs_optuna.csv (8x5)
      metric    fixed   optuna  optuna_minus_fixed interpretation
test_roc_auc 0.991375 0.990947           -0.000428   fixed better
 test_pr_auc 0.965900 0.968337            0.002437  optuna better
    test_ece 0.309514 0.277614           -0.031900  optuna better
   lodo_iscx 0.452788 0.548441            0.095653  optuna better
 lodo_usbvpn 0.480180 0.441648           -0.038532   fixed better
   lodo_vnat 0.980637 0.974032           -0.006605   fixed better
   lodo_mean 0.637868 0.654707            0.016839  optuna better
    lodo_min 0.452788 0.441648           -0.011140   fixed better
  metric: paper_audit_outputs\metrics\optuna_best_params_summary.json


WindowsPath('C:/Users/scoti/PycharmProjects/ai-vpn-firewall/paper_audit_outputs/metrics/optuna_best_params_summary.json')

In [13]:

# --- 4b. fixed vs optuna comparison plot + optimization history ---
fig,ax=plt.subplots(figsize=(9,5))
m=fixed_vs.metric.values; x=np.arange(len(m)); w=0.38
ax.bar(x-w/2,fixed_vs["fixed"],w,label="fixed")
ax.bar(x+w/2,fixed_vs["optuna"],w,label="optuna")
ax.axhline(0.5,color="k",ls="--",lw=0.7)
ax.set(title="Fixed vs Optuna (corrected runtime artifact)",xticks=x); ax.set_xticklabels(m,rotation=35,ha="right"); ax.legend()
save_fig(fig,"fixed_vs_optuna.png")

trials_p=OPT/"optuna_corrected_trials.csv"
if trials_p.exists():
    tr=pd.read_csv(trials_p).sort_values("number")
    tr["best_so_far"]=tr["value"].cummax()
    fig,ax=plt.subplots(figsize=(8,5))
    ax.plot(tr.number,tr.value,"o",alpha=0.5,label="trial val ROC-AUC")
    ax.plot(tr.number,tr.best_so_far,"-",color="red",label="best so far")
    ax.set(title="Optuna optimization history (pooled study, 60 trials)",xlabel="trial",ylabel="validation ROC-AUC"); ax.legend()
    save_fig(fig,"optuna_optimization_history.png")
else:
    mark_missing("4","optuna trials csv not found","optuna_corrected_trials.csv")


  figure: paper_audit_outputs\figures\fixed_vs_optuna.png (dpi=300)


  figure: paper_audit_outputs\figures\optuna_optimization_history.png (dpi=300)


## 5. Final robustness summary (diagnostic + runtime + Optuna)

In [14]:

rob_diag=prev.get("robustness_checks_diagnostic")
rows=[]
if rob_diag is not None:
    for _,r in rob_diag.iterrows():
        rows.append({"check":r.get("check"),"representation":"21-feature safe_core_plus_temporal",
            "model":r.get("model"),"setup":r.get("observation"),
            "training_domains":r.get("train_domains"),"validation_domains":r.get("val_domains"),
            "test_domains":r.get("test_domains"),
            "roc_auc":None,"pr_auc":None,
            "lodo_mean":r.get("lodo_mean_roc"),"lodo_min":r.get("lodo_min_roc"),
            "main_observation":r.get("observation"),"conclusion":r.get("conclusion"),
            "missing_or_limitations":"" if pd.notna(r.get("lodo_mean_roc")) else "component not available"})
else:
    mark_missing("5","diagnostic robustness table missing","Notebook 2 robustness_checks_diagnostic.csv")

# runtime fixed
rows.append({"check":"runtime fixed 12-feature LGBM","representation":"unified_relative_shape_v2 (12-feat)",
    "model":"LGBM(500,0.05,31)","setup":"pooled + LODO, isotonic calibration, val thresholds",
    "training_domains":"pooled train / source train (LODO)","validation_domains":"val split",
    "test_domains":"pooled test + held-out (LODO)","roc_auc":pooled_roc,"pr_auc":pooled_pr,
    "lodo_mean":lodo_mean,"lodo_min":lodo_min,
    "main_observation":"strong pooled, LODO collapses (iscx/usbvpn near/under 0.5)",
    "conclusion":"in-distribution strong; cross-dataset transfer fails",
    "missing_or_limitations":"simulation-only decision layer"})
# optuna
rows.append({"check":"runtime Optuna 12-feature LGBM","representation":"unified_relative_shape_v2 (12-feat)",
    "model":"LGBM(Optuna best)","setup":"60 trials TPE, val-AUC objective, source/val only",
    "training_domains":"pooled train / source train (LODO)","validation_domains":"val split",
    "test_domains":"pooled test + held-out (LODO)","roc_auc":opt_pooled["test_roc_auc"],"pr_auc":opt_pooled["test_pr_auc"],
    "lodo_mean":opt_lodo_mean,"lodo_min":opt_lodo_min,
    "main_observation":"pooled ~unchanged, ECE improved, lodo_min worse than fixed",
    "conclusion":"tuning does not resolve robustness; not an automatic replacement",
    "missing_or_limitations":"sensitivity check only"})
rob_full=pd.DataFrame(rows)
save_table_all(rob_full,"robustness_checks_full","Full robustness checks (diagnostic + runtime + Optuna)")
print(rob_full[["check","roc_auc","lodo_mean","lodo_min","conclusion"]].to_string(index=False))
rob_full


  table: paper_audit_outputs\tables\robustness_checks_full.csv (14x14)
                                        check  roc_auc  lodo_mean  lodo_min                                                       conclusion
                     baseline 21-feature LODO      NaN   0.582063  0.484982                               transfer fails (near/under chance)
                         rate-feature removal      NaN   0.561532  0.475100                               transfer fails (near/under chance)
feature pruning (no construction descriptors)      NaN   0.557356  0.454293                               transfer fails (near/under chance)
                                log transform      NaN   0.563297  0.491999                               transfer fails (near/under chance)
                             standard scaling      NaN   0.579005  0.491451                               transfer fails (near/under chance)
                               robust scaling      NaN   0.558836  0.467712        

,check,representation,model,setup,training_domains,validation_domains,test_domains,roc_auc,pr_auc,lodo_mean,lodo_min,main_observation,conclusion,missing_or_limitations
0,baseline 21-feature LODO,21-feature safe_core_plus_temporal,"XGBoost(300,5,0.1)",reference LODO,two source datasets,source val split,held-out dataset,NaN,NaN,0.582063,0.484982,reference LODO,transfer fails (near/under chance),
1,rate-feature removal,21-feature safe_core_plus_temporal,"XGBoost(300,5,0.1)",drop rate features,two source datasets,source val split,held-out dataset,NaN,NaN,0.561532,0.475100,drop rate features,transfer fails (near/under chance),
2,feature pruning (no construction descriptors),21-feature safe_core_plus_temporal,"XGBoost(300,5,0.1)",drop construction scale features,two source datasets,source val split,held-out dataset,NaN,NaN,0.557356,0.454293,drop construction scale features,transfer fails (near/under chance),
3,log transform,21-feature safe_core_plus_temporal,"XGBoost(300,5,0.1)",log compression of scale,two source datasets,source val split,held-out dataset,NaN,NaN,0.563297,0.491999,log compression of scale,transfer fails (near/under chance),
4,standard scaling,21-feature safe_core_plus_temporal,"XGBoost(300,5,0.1)",source-fit zscore,two source datasets,source val split,held-out dataset,NaN,NaN,0.579005,0.491451,source-fit zscore,transfer fails (near/under chance),
5,robust scaling,21-feature safe_core_plus_temporal,"XGBoost(300,5,0.1)",source-fit median/IQR,two source datasets,source val split,held-out dataset,NaN,NaN,0.558836,0.467712,source-fit median/IQR,transfer fails (near/under chance),
6,quantile normalization,21-feature safe_core_plus_temporal,"XGBoost(300,5,0.1)",rank->normal mapping,two source datasets,source val split,held-out dataset,NaN,NaN,0.554860,0.472640,rank->normal mapping,transfer fails (near/under chance),
7,whitening,21-feature safe_core_plus_temporal,"XGBoost(300,5,0.1)",decorrelate + unit variance,two source datasets,source val split,held-out dataset,NaN,NaN,0.425805,0.251990,decorrelate + unit variance,transfer fails (near/under chance),
8,PCA projection,21-feature safe_core_plus_temporal,"XGBoost(300,5,0.1)",linear dim reduction,two source datasets,source val split,held-out dataset,NaN,NaN,0.446846,0.350798,linear dim reduction,transfer fails (near/under chance),
9,covariate reweighting,21-feature safe_core_plus_temporal,NaN,component not present in project,NaN,NaN,NaN,NaN,NaN,NaN,NaN,component not present in project,MISSING / NEEDS MANUAL INPUT,component not available


## 6. Publication-ready replacement tables (CSV + Markdown + LaTeX)

In [15]:

# Re-emit prior diagnostic tables in all three formats with consistent naming.
def reemit(stem_in, basename_out, caption):
    df=prev.get(stem_in)
    if df is None:
        mark_missing("6",f"cannot re-emit {basename_out}: source {stem_in} missing","Notebook 1/2 output"); return
    save_table_all(df, basename_out, caption)

reemit("dataset_composition","pub_dataset_composition","Dataset composition (ISCX/USBVPN/VNAT)")
reemit("construction_scale_with_iqr","pub_construction_scale_iqr","Construction-scale descriptors with IQR")
reemit("dataset_fingerprinting_metrics","pub_dataset_fingerprinting","Dataset fingerprinting metrics")
reemit("feature_instability_detailed","pub_feature_instability","Feature direction-instability (SMD/rank)")
reemit("instability_verdict_features","pub_instability_verdict","Instability verdict with exact feature names")
reemit("feature_definitions_21","pub_feature_definitions_21","21-feature definitions")
reemit("flow_construction_audit","pub_flow_construction_loader_audit","Flow-construction / loader audit")

# improved intra vs cross performance table (merge intra + lodo)
intra=prev.get("intra_dataset_metrics"); lodo=prev.get("lodo_metrics")
if intra is not None and lodo is not None:
    ic=intra[["dataset","roc_auc","pr_auc"]].rename(columns={"roc_auc":"intra_roc_auc","pr_auc":"intra_pr_auc"})
    lc=lodo[["target","roc_auc","pr_auc"]].rename(columns={"target":"dataset","roc_auc":"cross_roc_auc","pr_auc":"cross_pr_auc"})
    perf=ic.merge(lc,on="dataset",how="outer")
    perf["roc_auc_gap"]=perf["intra_roc_auc"]-perf["cross_roc_auc"]
    perf["pr_auc_gap"]=perf["intra_pr_auc"]-perf["cross_pr_auc"]
    save_table_all(perf,"pub_intra_vs_cross_performance","Intra vs cross-dataset performance (21-feature)")
    print(perf.to_string(index=False))
else:
    mark_missing("6","intra/lodo metrics missing for combined performance table","Notebook 2 outputs")


  table: paper_audit_outputs\tables\pub_dataset_composition.csv (3x12)
  table: paper_audit_outputs\tables\pub_construction_scale_iqr.csv (15x5)
  table: paper_audit_outputs\tables\pub_dataset_fingerprinting.csv (2x10)
  table: paper_audit_outputs\tables\pub_feature_instability.csv (21x12)
  table: paper_audit_outputs\tables\pub_instability_verdict.csv (2x3)
  table: paper_audit_outputs\tables\pub_feature_definitions_21.csv (21x2)
  table: paper_audit_outputs\tables\pub_flow_construction_loader_audit.csv (3x18)
  table: paper_audit_outputs\tables\pub_intra_vs_cross_performance.csv (3x7)
dataset  intra_roc_auc  intra_pr_auc  cross_roc_auc  cross_pr_auc  roc_auc_gap  pr_auc_gap
   iscx       0.983444      0.974399       0.484982      0.283483     0.498462    0.690915
 usbvpn       0.982431      0.998698       0.548234      0.187624     0.434197    0.811074
   vnat       1.000000      1.000000       0.712974      0.109557     0.287026    0.890443


In [16]:

# model configuration tables, split into small readable tables
diag_cfg=pd.DataFrame([
 {"setting":"model","value":"XGBClassifier"},
 {"setting":"n_estimators","value":"300"},{"setting":"max_depth","value":"5"},
 {"setting":"learning_rate","value":"0.1"},{"setting":"tree_method","value":"hist"},
 {"setting":"objective","value":"binary:logistic (default)"},
 {"setting":"eval_metric","value":"logloss"},
 {"setting":"subsample","value":"1.0 (default)"},{"setting":"colsample_bytree","value":"1.0 (default)"},
 {"setting":"min_child_weight","value":"1 (default)"},{"setting":"gamma","value":"0.0 (default)"},
 {"setting":"reg_lambda","value":"1.0 (default)"},{"setting":"reg_alpha","value":"0.0 (default)"},
 {"setting":"scale_pos_weight","value":"1.0 (no class weighting)"},
 {"setting":"missing","value":"np.nan (native NaN handling)"},
 {"setting":"n_jobs","value":"0"},{"setting":"random_state","value":"42"},
 {"setting":"other_params","value":"XGBoost library defaults"},
 {"setting":"xgboost_version","value":"3.2.0"},
 {"setting":"split","value":"predefined capture-grouped train/val/test split column (seed 42)"},
 {"setting":"threshold","value":"validation Youden's J"}])
save_table_all(diag_cfg,"pub_model_config_diagnostic_xgb","Diagnostic 21-feature LODO model config (XGBoost)")

rt_cfg=pd.DataFrame([
 {"setting":"model","value":"LGBMClassifier"},{"setting":"objective","value":"binary"},
 {"setting":"n_estimators","value":"500"},{"setting":"learning_rate","value":"0.05"},
 {"setting":"num_leaves","value":"31"},{"setting":"min_child_samples","value":"20"},
 {"setting":"subsample","value":"0.8"},{"setting":"colsample_bytree","value":"0.8"},
 {"setting":"reg_alpha","value":"0.1"},{"setting":"reg_lambda","value":"0.1"},
 {"setting":"scale_pos_weight","value":f"{fm.get('build_stats',{}) and ''}n_neg/n_pos"},
 {"setting":"random_state","value":"42"},{"setting":"calibration","value":"isotonic (validation)"}])
save_table_all(rt_cfg,"pub_model_config_runtime_lgbm","Runtime 12-feature model config (LightGBM)")

opt_cfg=pd.DataFrame([{"setting":k,"value":str(v)} for k,v in opt_best_params.items()])
save_table_all(opt_cfg,"pub_model_config_optuna_best","Optuna best hyperparameters (pooled study)")

opt_setup=pd.DataFrame([
 {"setting":"trials","value":"60"},{"setting":"sampler","value":"TPESampler(seed=42)"},
 {"setting":"objective","value":"validation ROC-AUC"},{"setting":"direction","value":"maximize"},
 {"setting":"scale_pos_weight","value":opt_pooled.get("scale_pos_weight_choice")},
 {"setting":"best_val_auc","value":f"{opt_pooled['best_val_auc']:.5f}"}])
save_table_all(opt_setup,"pub_model_config_optuna_setup","Optuna search configuration")

fp_cfg=pd.DataFrame([
 {"setting":"model","value":"XGBClassifier (multiclass)"},
 {"setting":"objective","value":"multi:softprob"},{"setting":"num_class","value":"3"},
 {"setting":"n_estimators","value":"300"},{"setting":"max_depth","value":"5"},
 {"setting":"learning_rate","value":"0.1"},{"setting":"tree_method","value":"hist"},
 {"setting":"eval_metric","value":"mlogloss"},
 {"setting":"scale_pos_weight","value":"none (no class weighting)"},
 {"setting":"missing","value":"np.nan (native NaN handling)"},
 {"setting":"n_jobs","value":"0"},{"setting":"random_state","value":"42"},
 {"setting":"other_params","value":"XGBoost library defaults"},
 {"setting":"xgboost_version","value":"3.2.0"},
 {"setting":"split","value":"capture-level train/test (all flows of a capture in one split; zero capture overlap)"},
 {"setting":"evaluation","value":"held-out test captures only; single held-out split (not k-fold)"},
 {"setting":"uncertainty","value":"500-resample flow-level bootstrap 95% CI on macro-AUC"},
 {"setting":"imbalance_handling","value":"none; macro-averaged AUC/F1 reported"}])
save_table_all(fp_cfg,"pub_model_config_fingerprinting_xgb","Dataset-fingerprinting classifier config (XGBoost multiclass)")
print("model config tables written")


  table: paper_audit_outputs\tables\pub_model_config_diagnostic_xgb.csv (7x2)
  table: paper_audit_outputs\tables\pub_model_config_runtime_lgbm.csv (13x2)
  table: paper_audit_outputs\tables\pub_model_config_optuna_best.csv (15x2)


  table: paper_audit_outputs\tables\pub_model_config_optuna_setup.csv (6x2)
model config tables written


## 7. Publication-quality replacement figures (300 DPI)

In [17]:

# Figure A: improved pipeline diagram (schematic, no invented numbers)
fig,ax=plt.subplots(figsize=(12,7)); ax.axis("off")
def box(x,y,w,h,txt,fc):
    ax.add_patch(plt.Rectangle((x,y),w,h,fc=fc,ec="black",lw=1.2,alpha=0.9))
    ax.text(x+w/2,y+h/2,txt,ha="center",va="center",fontsize=9,wrap=True)
def arrow(x1,y1,x2,y2): ax.annotate("",xy=(x2,y2),xytext=(x1,y1),arrowprops=dict(arrowstyle="->",lw=1.3))
box(0.02,0.78,0.22,0.16,"Dataset loaders\nISCXVPN2016 / USBVPN / VNAT","#cfe8ff")
box(0.30,0.78,0.20,0.16,"Unified schema\n(flows + capture_id + label)","#cfe8ff")
box(0.58,0.78,0.20,0.16,"Capture-level split\n(70/15/15, seed=42)","#cfe8ff")
box(0.30,0.54,0.20,0.14,"Pooled evaluation\n(in-distribution)","#d8f5d0")
box(0.58,0.54,0.20,0.14,"Leave-one-dataset-out\n(cross-dataset)","#d8f5d0")
box(0.02,0.30,0.30,0.16,"Structural diagnostics\nfingerprinting / SMD instability /\nconstruction-scale / preprocessing","#fde9c8")
box(0.40,0.30,0.26,0.16,"Runtime 12-feature model\n(LightGBM + isotonic)\nvalidation-only thresholds","#f6d6e0")
box(0.72,0.30,0.24,0.16,"SIMULATION-ONLY layer\nPASS / FLAG_REVIEW /\nSIMULATED_BLOCK","#f3c0c0")
box(0.40,0.06,0.26,0.14,"Optuna sensitivity\n(source/val only, 60 trials)","#e7dff5")
arrow(0.24,0.86,0.30,0.86); arrow(0.50,0.86,0.58,0.86)
arrow(0.68,0.78,0.50,0.68); arrow(0.68,0.78,0.68,0.68)
arrow(0.40,0.54,0.30,0.46); arrow(0.50,0.54,0.53,0.46)
arrow(0.53,0.30,0.53,0.20); arrow(0.66,0.38,0.72,0.38)
ax.text(0.5,0.99,"Figure A — Audited pipeline (loaders → unified schema → split → evaluation → diagnostics → simulation-only runtime)",
        ha="center",fontsize=10,weight="bold")
save_fig(fig,"figA_pipeline_diagram.png")


  figure: paper_audit_outputs\figures\figA_pipeline_diagram.png (dpi=300)


WindowsPath('C:/Users/scoti/PycharmProjects/ai-vpn-firewall/paper_audit_outputs/figures/figA_pipeline_diagram.png')

In [18]:

# Figures B & C: intra vs cross ROC-AUC / PR-AUC (consistent naming, 0.5 baseline, CIs if present)
DS_LABEL={"iscx":"ISCXVPN2016","usbvpn":"USBVPN","vnat":"VNAT"}
intra=prev.get("intra_dataset_metrics"); lodo=prev.get("lodo_metrics")
if intra is not None and lodo is not None:
    order=["iscx","usbvpn","vnat"]; x=np.arange(len(order)); w=0.38
    iroc=[float(intra[intra.dataset==d].roc_auc.iloc[0]) for d in order]
    croc=[float(lodo[lodo.target==d].roc_auc.iloc[0]) for d in order]
    ipr=[float(intra[intra.dataset==d].pr_auc.iloc[0]) for d in order]
    cpr=[float(lodo[lodo.target==d].pr_auc.iloc[0]) for d in order]
    # CIs from intra table if available
    yerr=None
    if {"roc_auc_ci_lo","roc_auc_ci_hi"}.issubset(intra.columns):
        lo=[float(intra[intra.dataset==d].roc_auc_ci_lo.iloc[0]) for d in order]
        hi=[float(intra[intra.dataset==d].roc_auc_ci_hi.iloc[0]) for d in order]
        yerr=[np.array(iroc)-np.array(lo),np.array(hi)-np.array(iroc)]
    fig,ax=plt.subplots(figsize=(8,5))
    ax.bar(x-w/2,iroc,w,yerr=yerr,capsize=4,label="intra-dataset")
    ax.bar(x+w/2,croc,w,label="cross-dataset (LODO)")
    ax.axhline(0.5,color="k",ls="--",lw=0.8,label="random (0.5)")
    ax.set(title="Figure B — Intra vs Cross-dataset ROC-AUC (21-feature)",xticks=x,ylim=(0,1.05))
    ax.set_xticklabels([DS_LABEL[d] for d in order]); ax.legend()
    save_fig(fig,"figB_intra_vs_cross_roc_auc.png")

    fig,ax=plt.subplots(figsize=(8,5))
    ax.bar(x-w/2,ipr,w,label="intra-dataset"); ax.bar(x+w/2,cpr,w,label="cross-dataset (LODO)")
    ax.set(title="Figure C — Intra vs Cross-dataset PR-AUC (21-feature)",xticks=x,ylim=(0,1.05))
    ax.set_xticklabels([DS_LABEL[d] for d in order]); ax.legend()
    save_fig(fig,"figC_intra_vs_cross_pr_auc.png")
else:
    mark_missing("7","intra/lodo metrics missing for Figures B/C","Notebook 2 outputs")


  figure: paper_audit_outputs\figures\figB_intra_vs_cross_roc_auc.png (dpi=300)


  figure: paper_audit_outputs\figures\figC_intra_vs_cross_pr_auc.png (dpi=300)


In [19]:

# Figure D: SMD heatmap ; Figure E: preprocessing sensitivity matrix (rebuilt readable)
inst=prev.get("feature_instability_detailed")
if inst is not None and {"iscx_smd","usbvpn_smd","vnat_smd"}.issubset(inst.columns):
    mat=inst.set_index("feature")[["iscx_smd","usbvpn_smd","vnat_smd"]]
    fig,ax=plt.subplots(figsize=(6,9)); vmax=np.nanmax(np.abs(mat.values))
    im=ax.imshow(mat.values,cmap="coolwarm",vmin=-vmax,vmax=vmax,aspect="auto")
    ax.set(xticks=range(3),xticklabels=["ISCX","USBVPN","VNAT"],yticks=range(len(mat)),yticklabels=mat.index,
           title="Figure D — SMD heatmap (sign flip = direction instability)")
    for (i,j),v in np.ndenumerate(mat.values):
        if v==v: ax.text(j,i,f"{v:.2f}",ha="center",va="center",fontsize=7)
    fig.colorbar(im,ax=ax,label="SMD (VPN - nonVPN)"); save_fig(fig,"figD_smd_heatmap.png")
else:
    mark_missing("7","feature_instability_detailed missing/!smd cols for Figure D","Notebook 2 output")

sens=prev.get("preprocessing_sensitivity")
if sens is not None:
    flip_cols=[c for c in sens.columns if c.endswith("_smd_flip")]
    if flip_cols:
        m=sens.set_index("feature")[flip_cols]
        fig,ax=plt.subplots(figsize=(7,9)); ax.imshow(m.values,cmap="gray_r",vmin=0,vmax=1,aspect="auto")
        for (i,j),v in np.ndenumerate(m.values): ax.text(j,i,"X" if v else "",ha="center",va="center",color="white" if v else "black",fontsize=8)
        ax.set(xticks=range(len(flip_cols)),xticklabels=[c.replace("_smd_flip","") for c in flip_cols],
               yticks=range(len(m)),yticklabels=m.index,title="Figure E — Preprocessing sensitivity (X = SMD sign flip)")
        ax.tick_params(axis="x",rotation=25); save_fig(fig,"figE_preprocessing_sensitivity.png")
    else:
        mark_missing("7","preprocessing_sensitivity has no *_smd_flip columns for Figure E","Notebook 2 output")
else:
    mark_missing("7","preprocessing_sensitivity missing for Figure E","Notebook 2 output")
# Figures F (calibration) and G (fixed vs optuna) already saved as runtime_calibration_curve.png / fixed_vs_optuna.png
print("Figure F = runtime_calibration_curve.png ; Figure G = fixed_vs_optuna.png (already saved)")


  figure: paper_audit_outputs\figures\figD_smd_heatmap.png (dpi=300)


  figure: paper_audit_outputs\figures\figE_preprocessing_sensitivity.png (dpi=300)
Figure F = runtime_calibration_curve.png ; Figure G = fixed_vs_optuna.png (already saved)


## 8. Final paper-repair checklist

In [20]:

leak21=prev.get("nb2_split_leakage_check"); leak_pass = bool(leak21 is not None and (leak21[[c for c in leak21.columns if "overlap" in c]].values==0).all())
split_complete = prev.get("within_dataset_split_summary") is not None
cap_counts = prev.get("capture_statistics_extended") is not None
ci_avail = bool(intra is not None and "roc_auc_ci_lo" in intra.columns)
pr_avail = bool(intra is not None and "pr_auc" in intra.columns)
clean_avail = (OUT/"tables"/"cleaning_summary.csv").exists()
pktwin_doc = (OUT/"tables"/"flow_construction_audit.csv").exists()

ans = {
 "Are split details complete?": split_complete,
 "Are train/validation/test capture counts available?": cap_counts,
 "Did leakage checks pass?": leak_pass,
 "Are confidence intervals available?": ci_avail,
 "Are PR-AUC values available?": pr_avail,
 "Are cleaning/removal counts available?": clean_avail,
 "Are packet-window limits documented?": pktwin_doc,
 "Is diagnostic-vs-runtime artifact difference explained?": (TBL/"runtime_count_difference_explanation.csv").exists(),
 "Are runtime count differences explainable?": (TBL/"runtime_count_difference_explanation.csv").exists(),
 "Are unstable features categorized correctly?": prev.get("instability_verdict_features") is not None,
 "Are preprocessing-sensitive features separated from raw-space instability?": prev.get("preprocessing_sensitivity") is not None,
 "Are runtime/firewall claims clearly simulation-only?": True,
}
soften=[
 "Do not claim the runtime model is deployment-ready; it is a simulation-only decision layer.",
 "Do not claim cross-dataset robustness: LODO ROC-AUC collapses (min ~0.45-0.55 for ISCX/USBVPN).",
 "Do not present Optuna as an improvement/replacement: pooled ~unchanged, lodo_min worse than fixed.",
 "Do not claim USBVPN VPN blocking works in transfer: USBVPN LODO is near/under chance.",
 "Frame dataset fingerprinting (macro-AUC ~0.99) as evidence of structural dataset shift, not model quality.",
]
needs_manual=[m["what"] for m in MISSING]
lines=["# Final Paper-Repair Checklist\n",f"_Generated: {datetime.now().isoformat(timespec='seconds')}_\n",
 "\n## A. Values successfully computed\n",
 f"- pooled runtime ROC-AUC={pooled_roc:.5f}, PR-AUC={pooled_pr:.5f}, ECE={pooled_ece:.5f}, Brier={pooled_brier:.5f}\n",
 f"- runtime LODO: iscx={lodo_rt['iscx']:.5f}, usbvpn={lodo_rt['usbvpn']:.5f}, vnat={lodo_rt['vnat']:.5f}, mean={lodo_mean:.5f}, min={lodo_min:.5f}\n",
 f"- Optuna pooled ROC-AUC={opt_pooled['test_roc_auc']:.5f}, lodo_min={opt_lodo_min:.5f}\n",
 "\n## B. Plots successfully generated\n"]
for g in sorted(set(p for p in GENERATED if p.endswith(".png"))): lines.append(f"- {g}\n")
lines.append("\n## C. Tables successfully generated\n")
for g in sorted(set(p for p in GENERATED if p.endswith(".csv"))): lines.append(f"- {g}\n")
lines.append("\n## D. Missing information\n")
if MISSING:
    for m in MISSING: lines.append(f"- [{m['section']}] {m['what']} -> needs: {m['needed']}\n")
else: lines.append("- none\n")
lines.append("\n## E. Methodological warnings\n")
for s in soften: lines.append(f"- {s}\n")
lines.append("\n## F. Recommended paper edits / explicit answers\n")
for q,a in ans.items(): lines.append(f"- {q} **{'YES' if a else 'NO'}**\n")
lines.append("\n### Which values still need manual verification\n")
for n in (needs_manual or ["none"]): lines.append(f"- {n}\n")
p=LOG/"final_paper_repair_checklist.md"; p.write_text("".join(lines),encoding="utf-8")
GENERATED.append(str(p.relative_to(ROOT)))
print("checklist:",p.relative_to(ROOT))
for q,a in ans.items(): print(f"  {a and 'YES' or 'NO ':4} {q}")


checklist: paper_audit_outputs\logs\final_paper_repair_checklist.md
  YES  Are split details complete?
  YES  Are train/validation/test capture counts available?
  YES  Did leakage checks pass?
  YES  Are confidence intervals available?
  YES  Are PR-AUC values available?
  YES  Are cleaning/removal counts available?
  YES  Are packet-window limits documented?
  YES  Is diagnostic-vs-runtime artifact difference explained?
  YES  Are runtime count differences explainable?
  YES  Are unstable features categorized correctly?
  YES  Are preprocessing-sensitive features separated from raw-space instability?
  YES  Are runtime/firewall claims clearly simulation-only?


## 9. Final summary

In [21]:

comp0=prev.get("nb2_data_composition")
total_flows=int(comp0.flows.sum()) if comp0 is not None else None
total_caps=int(comp0.captures.sum()) if comp0 is not None else None
intra_roc={d:float(intra[intra.dataset==d].roc_auc.iloc[0]) for d in ["iscx","usbvpn","vnat"]} if intra is not None else {}
intra_pr={d:float(intra[intra.dataset==d].pr_auc.iloc[0]) for d in ["iscx","usbvpn","vnat"]} if intra is not None else {}
lodo_roc_d={d:float(lodo[lodo.target==d].roc_auc.iloc[0]) for d in ["iscx","usbvpn","vnat"]} if lodo is not None else {}
lodo_pr_d={d:float(lodo[lodo.target==d].pr_auc.iloc[0]) for d in ["iscx","usbvpn","vnat"]} if lodo is not None else {}
fp=prev.get("dataset_fingerprinting_metrics")
fp_macro={r["experiment"]:r["macro_auc"] for _,r in fp.iterrows()} if fp is not None else {}
inst=prev.get("feature_instability_detailed")
n_unstable=int((inst.instability_type=="verified raw-space direction instability").sum()) if inst is not None else None

summary={
 "datasets_audited":3,"total_flows_audited":total_flows,"total_captures_audited":total_caps,
 "diagnostic_intra_roc_auc":intra_roc,"diagnostic_intra_pr_auc":intra_pr,
 "diagnostic_lodo_roc_auc":lodo_roc_d,"diagnostic_lodo_pr_auc":lodo_pr_d,
 "dataset_fingerprint_macro_auc":fp_macro,"n_verified_unstable_features":n_unstable,
 "runtime_pooled_roc_auc":pooled_roc,"runtime_pooled_pr_auc":pooled_pr,
 "runtime_lodo_min":lodo_min,"optuna_lodo_min":opt_lodo_min,
 "n_missing_items":len(MISSING),
 "recommended_edits":["runtime = simulation-only, not deployment-ready",
   "cross-dataset robustness unresolved (LODO collapses)",
   "Optuna does not replace fixed model"],
}
save_metric(summary,"nb3_final_summary.json")
save_table(pd.DataFrame(MISSING) if MISSING else pd.DataFrame([{"section":"-","what":"none","needed":"-"}]),"nb3_missing_items.csv")
print(json.dumps(summary,indent=2,default=str))
print("\nALL GENERATED OUTPUTS:")
for g in sorted(set(GENERATED)): print("  ",g)
print("\ntotal generated this notebook:",len(set(GENERATED)))


  metric: paper_audit_outputs\metrics\nb3_final_summary.json
  table: paper_audit_outputs\tables\nb3_missing_items.csv (1x3)
{
  "datasets_audited": 3,
  "total_flows_audited": 72612,
  "total_captures_audited": 340,
  "diagnostic_intra_roc_auc": {
    "iscx": 0.9834439568227662,
    "usbvpn": 0.9824307500778088,
    "vnat": 1.0
  },
  "diagnostic_intra_pr_auc": {
    "iscx": 0.9743986513792072,
    "usbvpn": 0.9986982288050604,
    "vnat": 1.0
  },
  "diagnostic_lodo_roc_auc": {
    "iscx": 0.484981871636966,
    "usbvpn": 0.5482335645763357,
    "vnat": 0.7129736368407914
  },
  "diagnostic_lodo_pr_auc": {
    "iscx": 0.2834832520640069,
    "usbvpn": 0.1876243619991204,
    "vnat": 0.1095566093840004
  },
  "dataset_fingerprint_macro_auc": {
    "A_construction5": 0.9854574476185486,
    "B_full21": 0.9993161668163092
  },
  "n_verified_unstable_features": 16,
  "runtime_pooled_roc_auc": 0.9913752332786281,
  "runtime_pooled_pr_auc": 0.9659002974610761,
  "runtime_lodo_min": 0.45278